# Fine-Tuning BGE-Base Dense Bi-Encoder for Agricultural Extension RAG

This notebook fine-tunes `BAAI/bge-base-en-v1.5` for domain-specific agricultural dense retrieval using **Sentence Transformers v3**.

### Key Fixes Implemented in this Notebook:
1. **Preserves Native `[CLS]` Pooling:** Discards the custom mean-pooling layer that was drowning out query semantics with the instruction prefix.
2. **Prevents Representation Collapse:** Uses `MultipleNegativesRankingLoss` with deduplicated query pairs and `BatchSamplers.NO_DUPLICATES` to prevent in-batch negative collisions.
3. **Native SBERT Artifact:** Saves a complete `SentenceTransformer` package (`modules.json`, `1_Pooling`), ensuring seamless drop-in compatibility with the prototype.
4. **Built-in Sanity Verification:** Automatically tests distinct questions at the end to prove outputs are dynamic and relevant before downloading.

In [ ]:
# 1. Install modern dependencies
!pip install -q "sentence-transformers>=3.0" datasets accelerate


In [ ]:
# 2. Imports & Device Setup
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.training_args import BatchSamplers

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")


In [ ]:
# 3. Locate and Load Dataset (works in Google Colab, Kaggle, or locally)
CANDIDATE_DIRS = [
    Path("./"),
    Path("/content"),
    Path("/content/data"),
    Path("./data"),
    Path("/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers"),
    Path("/kaggle/input/agricultural-extension-rag-smart-retrieval-for-farmers"),
]

def find_file(filename: str) -> Path:
    for d in CANDIDATE_DIRS:
        p = d / filename
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Could not find '{filename}'. If on Colab, please upload '{filename}' via the files sidebar."
    )

docs_path = find_file("documents.csv")
train_queries_path = find_file("train_queries.csv")
qrels_path = find_file("qrels_train.csv")

print(f"Loading documents from: {docs_path}")
df_docs = pd.read_csv(docs_path, index_col="document_id")
train_queries_df = pd.read_csv(train_queries_path)
qrels_train_df = pd.read_csv(qrels_path)

print(f"Loaded {len(df_docs)} documents, {len(train_queries_df)} queries, {len(qrels_train_df)} relevance labels.")


In [ ]:
# 4. Flatten Documents (exact competition template parity)
def create_search_content(row):
    crop = str(row.get("crop", " ")).strip()
    country = str(row.get("country", " ")).strip()
    title = str(row.get("title", " ")).strip()
    text = str(row.get("text", " ")).strip()
    source = str(row.get("source", " ")).strip()
    return f"Crop {crop} | Country {country} | Title {title} | Text {text} | Source {source}"

df_docs["search_text"] = df_docs.apply(create_search_content, axis=1)
doc_dict = df_docs["search_text"].to_dict()
print(f"Indexed {len(doc_dict)} documents into search texts.")


In [ ]:
# 5. Construct Clean Contrastive Training Samples
# BGE requires the asymmetric instruction prefix for queries
BGE_PREFIX = "Represent this sentence for searching relevant passages: "

tq_id_col = "QueryId" if "QueryId" in train_queries_df.columns else train_queries_df.columns[0]
tq_text_col = "Query" if "Query" in train_queries_df.columns else train_queries_df.columns[1]
query_dict = dict(zip(train_queries_df[tq_id_col], train_queries_df[tq_text_col]))

qrels_qid_col = "QueryId" if "QueryId" in qrels_train_df.columns else qrels_train_df.columns[0]
qrels_did_col = "DocumentId" if "DocumentId" in qrels_train_df.columns else qrels_train_df.columns[1]

records = []
for q_id, group in qrels_train_df.groupby(qrels_qid_col):
    if q_id not in query_dict:
        continue
    q_text = BGE_PREFIX + str(query_dict[q_id]).strip()
    
    # Primary positives (relevance >= 2, or fallback to any positive)
    pos_docs = group[group["relevance"] >= 2][qrels_did_col].tolist()
    if not pos_docs:
        pos_docs = group[group["relevance"] > 0][qrels_did_col].tolist()
    if not pos_docs:
        continue
        
    neg_docs = group[group["relevance"] == 0][qrels_did_col].tolist()
    
    # Pair each distinct positive without excessive query duplication
    for i, pos_id in enumerate(pos_docs):
        if pos_id not in doc_dict:
            continue
        entry = {
            "anchor": q_text,
            "positive": doc_dict[pos_id],
        }
        if neg_docs:
            neg_id = neg_docs[i % len(neg_docs)]
            if neg_id in doc_dict:
                entry["negative"] = doc_dict[neg_id]
        records.append(entry)

train_dataset = Dataset.from_list(records)
print(f"Generated {len(train_dataset)} clean contrastive training samples.")
print("Sample record:", records[0])


In [ ]:
# 6. Initialize BGE Base Model
# SentenceTransformer natively preserves [CLS] pooling from BAAI/bge-base-en-v1.5
model_name = "BAAI/bge-base-en-v1.5"
model = SentenceTransformer(model_name, device=device)
print(f"Loaded base model: {model_name}")


In [ ]:
# 7. Fine-Tune with MultipleNegativesRankingLoss & SBERT Trainer
train_loss = losses.MultipleNegativesRankingLoss(model)
output_dir = "./fine_tuned_bge_base_agri"

training_args = SentenceTransformerTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # Prevents in-batch false negative collisions
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    loss=train_loss,
)

print("Starting fine-tuning...")
trainer.train()

# Save the complete model package with modules.json and pooling configs
model.save_pretrained(output_dir)
print(f"Fine-tuning complete! Model saved to: {output_dir}")


In [ ]:
# 8. Sanity Check: Verify Dynamic, Relevant Retrieval (No Static Output!)
test_queries = [
    "My maize leaves are turning yellow, what should I do?",
    "When is the best time to plant rice in northern Nigeria?",
    "How do I control fall armyworm without chemicals?",
]

corpus_ids = list(doc_dict.keys())
corpus_texts = list(doc_dict.values())

print("Encoding 695 corpus documents with fine-tuned model...")
doc_embeddings = model.encode(
    corpus_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_tensor=True,
)

print("\n--- Verification Results ---")
for q in test_queries:
    q_emb = model.encode(BGE_PREFIX + q, normalize_embeddings=True, convert_to_tensor=True)
    scores = torch.matmul(doc_embeddings, q_emb)
    top_indices = torch.topk(scores, k=3).indices.cpu().tolist()
    
    print(f"\nQuery: '{q}'")
    for rank, idx in enumerate(top_indices, 1):
        doc_id = corpus_ids[idx]
        doc_title = df_docs.loc[doc_id, "title"]
        print(f"  {rank}. [{doc_id}] {doc_title}")


In [ ]:
# 9. (Optional) Zip Model for Easy Download from Colab
!zip -r fine_tuned_bge_base_agri.zip fine_tuned_bge_base_agri/
print("Model zipped to fine_tuned_bge_base_agri.zip. Ready to download or push to Hugging Face!")
